# EDA — jointure rendements × climat

Exploration de la table consolidée (`data/processed/rendements_climat.csv`).
La logique de jointure et d'anomalies vit dans `src/agri_climat/data/join.py` :
ce notebook l'appelle, il ne la recopie pas.

Prérequis : `python -m agri_climat join` (ou `run`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from agri_climat.data.join import (
    climate_extremes,
    coverage_by_crop_geo,
    missingness,
    yield_summary_by_crop,
)
from agri_climat.paths import processed_dir

csv_path = processed_dir() / "rendements_climat.csv"
if not csv_path.exists():
    raise FileNotFoundError(
        f"{csv_path} introuvable — lancer : python -m agri_climat join"
    )

joined = pd.read_csv(csv_path)
joined.head()

## Couverture et manquants

Nombre d'années par culture et territoire ; colonnes encore incomplètes.

In [ ]:
coverage_by_crop_geo(joined)

In [ ]:
missingness(joined)

## Rendements (Wallonie) et extrêmes climatiques

Les z-scores sont calculés **par territoire** sur 2000–2024.

In [ ]:
yield_summary_by_crop(joined)

In [ ]:
climate_extremes(joined, geo="BE3")

## Aperçu visuel (Wallonie)

Figures d'exploration : les graphiques « portfolio » seront exportés en Feature 4.

In [ ]:
# Une ligne climat par année (le climat ne dépend pas de la culture).
wallonie = joined[joined["geo"] == "BE3"]
climat = wallonie.drop_duplicates(subset=["year"]).sort_values("year")
froment = wallonie[wallonie["crop_code"] == "C1110"].sort_values("year")

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
axes[0].plot(froment["year"], froment["yield_t_ha"], marker="o")
axes[0].set_ylabel("Rendement froment (t/ha)")
axes[1].bar(climat["year"], climat["precip_growing_mm_z"], color="steelblue")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Pluie saison (z-score)")
axes[1].set_xlabel("Année")
fig.tight_layout()
plt.show()